In [ ]:
# Installations (Comment out if already installed)
!pip install qiskit
!pip install pylatexenc

In [ ]:
# Imports
import os
from qiskit import QuantumCircuit, QuantumRegister, qpy

In [ ]:
def esop_to_circuit(file_path, verbose=False):
  """
  Realizes an Exclusive-or Sum of Products (ESOP) expression as a quantum circuit.

  Args:
    file_path (str): The file path (.esop) containing the function as an ESOP expression.
    verbose (bool): Prints information and shows progress if True.

  Returns:
    qc (QuantumCircuit): The resulting quantum circuit.
  """
  num_inputs = None
  num_outputs = None
  cubes = []

  # Read lines in the file
  with open(file_path, 'r') as f:
    lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    for line in lines:
      parts = line.split()

      # Parse info
      if parts[0] == '.i':  # Inputs
        num_inputs = int(parts[1])
        if verbose: print(f"Number of inputs: {num_inputs}")

      elif parts[0] == '.o':  # Outputs
        num_outputs = int(parts[1])
        if verbose: print(f"Number of outputs: {num_outputs}")

        if num_outputs > 1:
          print("Error: more than 1 output is not supported")
          return

      elif not parts[0].startswith('.') and parts[-1] == '1':  # Cubes
        cubes.append(parts[0])

      if num_inputs is None:
        print("Error: the number of inputs is not specified")
        return
      elif num_inputs is None:
        print("Error: the number of outputs is not specified")
        return

  if verbose: print(f"Cubes: {cubes}")

  # Build circuit
  qc = QuantumCircuit(num_inputs + num_outputs)
  target_qubit = qc.num_qubits - 1
  negated_qubits = set()

  for cube in cubes:  # Realizes a Toffoli gate for each cube
    control_qubits = []

    # Find control qubits and place NOT gates
    for i, literal in enumerate(cube):
      if literal == '1':
        control_qubits.append(i)
        if i in negated_qubits:
          negated_qubits.remove(i)
          qc.x(i)
          if verbose: print(f">> Added NOT gate to qubit {i} (Currently negated qubits: {negated_qubits})")

      elif literal == '0':
        control_qubits.append(i)
        if i not in negated_qubits:
          negated_qubits.add(i)
          qc.x(i)
          if verbose: print(f">> Added NOT gate to qubit {i} (Currently negated qubits: {negated_qubits})")

    # Adds Toffoli
    qc.mcx(control_qubits, target_qubit)
    if verbose: print(f"> Added Toffoli with control qubits {control_qubits}")

  with open(f"{os.path.splitext(file_path)[0]}-esop.qpy", "wb") as f:
    qpy.dump(qc, f)
    if verbose: print(f"Circuit saved to {os.path.splitext(file_path)[0]}-esop.qpy")

  return qc

# Run on Benchmarks
---

In [ ]:
# Convert .esop file to a circuit for a single benchmark
benchmark = "sam_ex1"  # <-- Change
qc = esop_to_circuit(f"esop/{benchmark}.esop")

In [ ]:
# Draw circuit
qc.draw(output='mpl')

In [ ]:
# Convert all benchmarks to circuits
benchmarks = ["sam_ex1", "sam_ex2", "sam_ex3", "sam_ex4", "sam_ex5",
              "rd53f1", "rd53f2", "rd73f1", "rd73f3", "rd84f1", "rd84f3", "rd84f4",
              "sym6_63", "sym9_71", "sym10_207", "co14_135"]

for benchmark in benchmarks:
  esop_to_circuit(f"/esop/{benchmark}.esop")